# NIH ChestX-ray14 preprocessing

This notebook creates the NIH federated client as metadata-only CSV files. It keeps clean target positives and explicit `No Finding` negatives, excludes ambiguous label combinations, preserves patient IDs, and splits whole patients. Images are never copied.

In [ ]:
from pathlib import Path
import json, sys, platform
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir() and Path('/content').exists():
    PROJECT_ROOT = Path('/content') / 'Project Repo'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from dataset_utils import nih_clean_binary_labels, standardize_nih
from preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary,
                                  require_directory, require_file, sample_clean_cases, save_splits,
                                  save_summary, set_reproducibility, split_by_patient)

set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, 'numpy:', np.__version__)

# Change only these paths for Drive, local storage, or another mounted directory.
NIH_ROOT = PROJECT_ROOT / 'data' / 'NIH'
METADATA_CSV = NIH_ROOT / 'Data_Entry_2017.csv'
IMAGE_DIR = NIH_ROOT / 'images'
OUTPUT_DIR = PROJECT_ROOT / 'processed' / 'client_01_NIH'
print({'NIH_ROOT': NIH_ROOT, 'METADATA_CSV': METADATA_CSV, 'IMAGE_DIR': IMAGE_DIR, 'OUTPUT_DIR': OUTPUT_DIR})

## Load and inspect
NIH `Finding Labels` is pipe-separated. The clean binary policy is deliberately conservative: a target-containing study is positive; exactly `No Finding` is negative; every other combination is excluded.

In [ ]:
metadata_path = require_file(METADATA_CSV, 'NIH metadata CSV')
image_dir = require_directory(IMAGE_DIR, 'NIH image directory')
nih = pd.read_csv(metadata_path)
print('images:', len(nih))
print('unique patients:', nih['Patient ID'].nunique())
print('columns:', nih.columns.tolist())
display(nih.isna().sum().sort_values(ascending=False).head(20).to_frame('missing'))
display(nih['Finding Labels'].value_counts(dropna=False).head(20).to_frame('label_count'))
clean, counts = nih_clean_binary_labels(nih, TARGET_DISEASE)
print('Counts before filtering:', counts)
print('Clean rows:', len(clean))

In [ ]:
standardized = standardize_nih(clean, image_dir)
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)
print('Rows after optional sampling:', len(standardized))
display(standardized.head())
print(standardized['target'].value_counts().sort_index())
splits = split_by_patient(standardized, seed=RANDOM_SEED)
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient leakage checks: passed')
print('study leakage checks: passed where study_id exists')
save_splits(splits, OUTPUT_DIR)
summary = build_summary(splits, 'NIH', {'target_disease': TARGET_DISEASE, 'random_seed': RANDOM_SEED, 'max_positive_samples': MAX_POSITIVE_SAMPLES, 'max_negative_samples': MAX_NEGATIVE_SAMPLES, 'filter_counts_before_filtering': counts, 'official_split_used': False})
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved:', sorted(path.name for path in OUTPUT_DIR.glob('*')))